# Hospital length-of-stay regression: model comparison and SHAP

This notebook compares four regression algorithms, selects the best model using **validation MAE only**, evaluates that locked model on the test set, and explains it with the installed `shap` package. It assumes the 1,000 synthetic profiles and task table are in `D:\\CTSA`. Synthetic educational data only; not for clinical decisions.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score

RANDOM_SEED = 42
# Locate the repository root whether Jupyter was started in the repo root or in notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "data" / "profiles_1000"
PROFILE_PATH = DATA_ROOT / 'synthetic_patient_profiles_1000.csv'
TASK_PATH = DATA_ROOT / 'length_of_stay_regression_dataset.csv'

profiles = pd.read_csv(PROFILE_PATH)
task = pd.read_csv(TASK_PATH)
assert len(profiles) == len(task) == 1000
print('Profiles:', profiles.shape, '| Regression table:', task.shape)
print('SHAP version:', shap.__version__)
profiles.head()

## Leakage-safe predictors and supplied splits

The task table already excludes mortality, disease class, ICU admission, and `log1p_length_of_stay`. The synthetic ID and split label are also excluded from predictors. Preprocessing is fitted only on training data.

In [ ]:
TARGET = 'length_of_stay_days'
NON_FEATURES = {TARGET, 'split', 'synthetic_patient_id'}
feature_cols = [c for c in task.columns if c not in NON_FEATURES]

train = task.loc[task['split'].eq('train')].copy()
validation = task.loc[task['split'].eq('validation')].copy()
test = task.loc[task['split'].eq('test')].copy()

X_train, y_train = train[feature_cols], train[TARGET]
X_validation, y_validation = validation[feature_cols], validation[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]

assert set(train.synthetic_patient_id).isdisjoint(validation.synthetic_patient_id)
assert set(train.synthetic_patient_id).isdisjoint(test.synthetic_patient_id)
assert set(validation.synthetic_patient_id).isdisjoint(test.synthetic_patient_id)
print({'train': len(train), 'validation': len(validation), 'test': len(test)})
y_train.describe()

## Preprocessing and four candidate models

The candidates offer complementary inductive biases: regularized linear (`Ridge`), sparse linear (`ElasticNet`), bagged nonlinear trees (`RandomForest`), and boosted nonlinear trees (`HistGradientBoosting`). One-hot output is dense so every candidate and SHAP explainer receives a consistent matrix.

In [ ]:
numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = [c for c in feature_cols if c not in numeric_features]

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical_features),
], verbose_feature_names_out=False)

candidate_models = {
    'Ridge': Ridge(alpha=5.0),
    'ElasticNet': ElasticNet(alpha=0.02, l1_ratio=0.30, max_iter=20_000, random_state=RANDOM_SEED),
    'Random forest': RandomForestRegressor(
        n_estimators=400, min_samples_leaf=3, max_features=0.75,
        random_state=RANDOM_SEED, n_jobs=-1),
    'Histogram gradient boosting': HistGradientBoostingRegressor(
        learning_rate=0.05, max_iter=300, max_leaf_nodes=20,
        l2_regularization=1.0, random_state=RANDOM_SEED),
}
candidate_models

## Fit, compare, and select using validation MAE

Lower validation MAE wins. RMSE, median absolute error, and R² are shown for context but do not influence selection.

In [ ]:
def regression_metrics(y_true, y_pred):
    y_pred = np.maximum(np.asarray(y_pred), 0)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'Median_AE': median_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }

fitted_models = {}
validation_rows = []
for model_name, estimator in candidate_models.items():
    pipeline = Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', clone(estimator)),
    ])
    pipeline.fit(X_train, y_train)
    fitted_models[model_name] = pipeline
    prediction = pipeline.predict(X_validation)
    validation_rows.append({'model': model_name, **regression_metrics(y_validation, prediction)})

validation_results = pd.DataFrame(validation_rows).sort_values('MAE').reset_index(drop=True)
best_model_name = validation_results.loc[0, 'model']
best_pipeline = fitted_models[best_model_name]
print('Selected model:', best_model_name)
validation_results.round(3)

## Final held-out test evaluation

The test split is evaluated only after model selection. Predictions are clipped at zero because negative hospital stays are impossible.

In [ ]:
test_prediction = np.maximum(best_pipeline.predict(X_test), 0)
test_results = pd.DataFrame([{'model': best_model_name, **regression_metrics(y_test, test_prediction)}])
display(test_results.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, test_prediction, alpha=0.65)
limit = max(float(y_test.max()), float(test_prediction.max()))
axes[0].plot([0, limit], [0, limit], '--', color='black')
axes[0].set(xlabel='Observed LOS (days)', ylabel='Predicted LOS (days)',
            title=f'Observed vs predicted: {best_model_name}')
residuals = np.asarray(y_test) - test_prediction
axes[1].scatter(test_prediction, residuals, alpha=0.65)
axes[1].axhline(0, linestyle='--', color='black')
axes[1].set(xlabel='Predicted LOS (days)', ylabel='Residual (observed − predicted)',
            title='Residual diagnostic')
fig.tight_layout()
plt.show()

## SHAP explanation of the selected model

SHAP is applied to the fitted estimator after the fitted preprocessing transformation. `TreeExplainer` is used for a tree-based winner and `LinearExplainer` for a linear winner. To keep workshop runtime modest, explanations use at most 100 background training rows and 150 test rows. One-hot names come from the fitted transformer. SHAP associations explain this fitted synthetic-data model; they are not causal effects or clinical evidence.

In [ ]:
fitted_preprocessor = best_pipeline.named_steps['preprocess']
fitted_estimator = best_pipeline.named_steps['model']
feature_names = fitted_preprocessor.get_feature_names_out()

X_train_transformed = np.asarray(fitted_preprocessor.transform(X_train), dtype=float)
X_test_transformed = np.asarray(fitted_preprocessor.transform(X_test), dtype=float)
rng = np.random.default_rng(RANDOM_SEED)
background_indices = rng.choice(len(X_train_transformed), size=min(100, len(X_train_transformed)), replace=False)
explain_indices = np.arange(min(150, len(X_test_transformed)))
background = X_train_transformed[background_indices]
X_explain = X_test_transformed[explain_indices]

if best_model_name in {'Random forest', 'Histogram gradient boosting'}:
    explainer = shap.TreeExplainer(fitted_estimator)
    shap_values = explainer(X_explain)
else:
    explainer = shap.LinearExplainer(fitted_estimator, background)
    shap_values = explainer(X_explain)

shap_values.feature_names = list(feature_names)
print('Explainer:', type(explainer).__name__)
print('Explained rows:', X_explain.shape[0], '| transformed features:', X_explain.shape[1])

In [ ]:
# Global direction and magnitude: each dot is one synthetic test profile.
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title(f'SHAP summary — {best_model_name}')
plt.tight_layout()
plt.show()

# Global mean absolute contribution.
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title(f'Mean absolute SHAP importance — {best_model_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Dependence plot for the most influential transformed feature.
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top_feature_index = int(np.argmax(mean_abs_shap))
top_feature_name = feature_names[top_feature_index]
shap.plots.scatter(shap_values[:, top_feature_index], color=shap_values, show=False)
plt.title(f'SHAP dependence: {top_feature_name}')
plt.tight_layout()
plt.show()

# Local explanation for one synthetic test profile.
shap.plots.waterfall(shap_values[0], max_display=15, show=False)
plt.title('Local SHAP explanation: first synthetic test profile')
plt.tight_layout()
plt.show()

## Interpretation checklist

- Positive SHAP values push the predicted LOS upward; negative values push it downward.
- The beeswarm combines effect magnitude and direction across synthetic test profiles.
- One-hot categories appear as separate transformed features.
- The waterfall explains one prediction relative to the model's expected prediction.
- Explanations describe model behavior on synthetic workshop data and must not be interpreted as causal or clinically validated.